In [1]:
import sys
sys.path.append("../")

from typing import TypedDict

from langgraph.graph import (
    StateGraph,
    END
)

from langchain_ollama import ChatOllama

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [3]:
class GraphState(TypedDict):

    question: str

    rewritten_question: str

    documents: list

    context: str

    answer: str

In [4]:
def rewrite_question(state):

    question = state["question"]

    prompt = f"""
Rewrite the following question to improve document retrieval.

Question:

{question}
"""

    rewritten = llm.invoke(prompt).content

    state["rewritten_question"] = rewritten

    return state

In [16]:

def retrieve_documents(state):

    query = state["rewritten_question"]

    docs = reranker.search(query)

    state["documents"] = docs

    return state

In [17]:
def build_context(state):

    docs = state["documents"]

    context = format_docs(docs)

    state["context"] = context

    return state

In [18]:
def generate_answer(state):

    prompt = f"""
You are an Enterprise AI Assistant.

Context

{state["context"]}

Question

{state["question"]}

Answer only using the context.
"""

    response = llm.invoke(prompt)

    state["answer"] = response.content

    return state

In [19]:
workflow = StateGraph(GraphState)

In [20]:
workflow.add_node(
    "rewrite",
    rewrite_question
)

workflow.add_node(
    "retrieve",
    retrieve_documents
)

workflow.add_node(
    "context",
    build_context
)

workflow.add_node(
    "generate",
    generate_answer
)

In [21]:
workflow.set_entry_point(
    "rewrite"
)

workflow.add_edge(
    "rewrite",
    "retrieve"
)

workflow.add_edge(
    "retrieve",
    "context"
)

workflow.add_edge(
    "context",
    "generate"
)

workflow.add_edge(
    "generate",
    END
)

In [22]:
graph = workflow.compile()

In [ ]:
result = graph.invoke(
    {
        "question":"Explain radar calibration."
    }
)

print(result["answer"])

AttributeError: module 'src.rerankers.reranker' has no attribute 'search'